# Random Forest & Logistic Regression Models

Train and evaluate Random Forest and Logistic Regression classifiers for bird species classification.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
import os
import numpy as np
import joblib
from pathlib import Path

DATA_DIR = Path("./processed_data")
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

if not npz_file.exists():
    print("Feature file not found. Trying single feature matrix fallback...")
    npz_file = DATA_DIR / "hog_features.npz"

if not npz_file.exists():
    raise FileNotFoundError("Processed feature file not found. Please run notebooks/02_Feature_Extraction.ipynb first!")

data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape}")
print(f"  Validation set: {X_val.shape}")
print(f"  Testing set   : {X_test.shape}")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Train Baseline & Tuned Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- Training Baseline Random Forest ---")
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, rf_baseline.predict(X_test))
print(f"Baseline Random Forest Test Accuracy: {baseline_acc * 100:.2f}%")

print("\n--- Hyperparameter Tuning with GridSearchCV ---")
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 15, 25],
    'max_features': ['sqrt', 'log2']
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search_rf.fit(X_train, y_train)
print(f"Best Random Forest Parameters: {grid_search_rf.best_params_}")
print(f"Best Cross-Val Score       : {grid_search_rf.best_score_ * 100:.2f}%")

## 2. Evaluate & Save Random Forest Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
final_acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Final Tuned Random Forest Test Accuracy: {final_acc_rf * 100:.2f}%")

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_rf, MODELS_DIR / "random_forest_model.pkl")
print("Random Forest model saved successfully!")

print("\nClassification Report (Random Forest):\n")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

plt.figure(figsize=(10, 8))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Optional: Train & Tune Logistic Regression Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression

print("--- Training & Tuning Logistic Regression ---")
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'max_iter': [1000]
}

grid_search_lr = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search_lr.fit(X_train, y_train)

best_lr = grid_search_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
final_acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Best Logistic Regression Parameters: {grid_search_lr.best_params_}")
print(f"Tuned Logistic Regression Accuracy  : {final_acc_lr * 100:.2f}%")

joblib.dump(best_lr, MODELS_DIR / "logistic_regression_model.pkl")
print("Logistic Regression model saved successfully!")

print("\nClassification Report (Logistic Regression):\n")
print(classification_report(y_test, y_pred_lr, target_names=class_names))